### Folder Structure

```
catchlog/
├── data/
│   ├── images/     ← .jpg files (UUID filenames)
│   └── labels/
│       ├── foid_labels_bbox_v012.csv
│       └── foid_labels_bbox_v012_freq.csv
└── output/         ← adapter weights land here
```


In [ ]:
# run once then comment out
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.47.0 peft bitsandbytes accelerate datasets pillow pandas tqdm

Quick Cuda Check


In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("NO GPU -- something is wrong, fix this first")

Config


In [ ]:
import os
from pathlib import Path

# paths
IMAGES_DIR = "data/images"
LABELS_CSV = "data/labels/foid_labels_bbox_v012.csv"
OUTPUT_DIR = "output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "catchlog-lora-adapter")
PROCESSED_DIR = os.path.join(OUTPUT_DIR, "processed")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# the 12 species we care about
# key = exact label_l1 value from the CSV
TARGET_SPECIES = {
    "Albacore":               {"name": "ALB",    "status": "legal"},
    "Yellowfin tuna":         {"name": "YFT",    "status": "legal"},
    "Bigeye tuna":            {"name": "BET",    "status": "legal_quota"},
    "Skipjack tuna":          {"name": "SKJ",    "status": "legal"},
    "Opah":                   {"name": "LAG",    "status": "legal"},
    "Mahi mahi":              {"name": "DOL",    "status": "legal"},
    "Shark":                  {"name": "SHARK",  "status": "bycatch"},
    "Thresher shark":         {"name": "SHARK",  "status": "bycatch"},
    "Swordfish":              {"name": "BILL",   "status": "legal_regulated"},
    "Striped marlin":         {"name": "BILL",   "status": "protected"},
    "Blue marlin":            {"name": "BILL",   "status": "protected"},
    "Black marlin":           {"name": "BILL",   "status": "protected"},
    "Shortbill spearfish":    {"name": "BILL",   "status": "legal_regulated"},
    "Indo Pacific sailfish":  {"name": "BILL",   "status": "protected"},
    "Oilfish":                {"name": "OTH",    "status": "legal"},
    "Wahoo":                  {"name": "OTH",    "status": "legal"},
    "Unknown":                {"name": "OTH",    "status": "unknown"},
    "Long snouted lancetfish": {"name": "OTH",   "status": "legal"},
    "Great barracuda":        {"name": "OTH",    "status": "legal"},
    "Sickle pomfret":         {"name": "OTH",    "status": "legal"},
    "Pelagic stingray":       {"name": "PLS",    "status": "bycatch"},
    "Mola mola":              {"name": "OTH",    "status": "bycatch"},
    "Pomfret":                {"name": "OTH",    "status": "legal"},
    "Rainbow runner":         {"name": "OTH",    "status": "legal"},
    "Snake mackerel":         {"name": "OTH",    "status": "legal"},
    "Roudie scolar":          {"name": "OIL",    "status": "legal"},
    "No fish":                {"name": "NoF",    "status": "none"},
    "Human":                  {"name": "HUMAN",  "status": "none"},
}
# training params
MAX_SAMPLES_PER_SPECIES = 600   # cap so albacore doesnt dominate (it has 33k lol)
VAL_SPLIT = 0.1
MODEL_ID = "google/paligemma2-3b-pt-224"
NUM_EPOCHS = 3
BATCH_SIZE = 4                   # rtx 6000 can handle this easy
GRADIENT_ACCUMULATION = 2
LEARNING_RATE = 2e-5
LORA_RANK = 8
MAX_NEW_TOKENS = 256

print(f"targeting {len(TARGET_SPECIES)} species")
print(f"model: {MODEL_ID}")
print(f"max {MAX_SAMPLES_PER_SPECIES} samples per species")

Exploration


In [ ]:
import pandas as pd

df = pd.read_csv(LABELS_CSV)
print(f"total annotations: {len(df)}")
print(f"columns: {list(df.columns)}")
print()
print("--- species breakdown ---")
print(df['label_l1'].value_counts().to_string())

### Target Species & Match Images

Keep the 12 species we want, and check which images actually exist on disk


In [ ]:
from collections import defaultdict

# filter to our species
df_filtered = df[df['label_l1'].isin(TARGET_SPECIES.keys())].copy()
print(f"after filtering: {len(df_filtered)} annotations")

# scan for actual image files on disk
print(f"\nscanning {IMAGES_DIR}...")
available_images = {}
for f in os.listdir(IMAGES_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
        img_id = os.path.splitext(f)[0]
        available_images[img_id] = os.path.join(IMAGES_DIR, f)

print(f"found {len(available_images)} images on disk")

# only keep annotations where the image actually exists
df_filtered = df_filtered[df_filtered['img_id'].isin(available_images.keys())]
print(f"annotations with matching images: {len(df_filtered)}")

# quick check
print(f"\nspecies we're using:")
for sp, count in df_filtered['label_l1'].value_counts().items():
    print(f"  {sp}: {count}")

Conversion to PaliGem format


In [ ]:
import json
import random
from PIL import Image
from tqdm import tqdm

random.seed(42)

def to_paligemma_locs(x_min, x_max, y_min, y_max, img_w, img_h):
    """pixel bbox -> paligemma normalized loc tokens (0-1023)"""
    ly1 = min(1023, max(0, int((y_min / img_h) * 1023)))
    lx1 = min(1023, max(0, int((x_min / img_w) * 1023)))
    ly2 = min(1023, max(0, int((y_max / img_h) * 1023)))
    lx2 = min(1023, max(0, int((x_max / img_w) * 1023)))
    return f"<loc{ly1:04d}><loc{lx1:04d}><loc{ly2:04d}><loc{lx2:04d}>"

# group annotations by image
image_annotations = defaultdict(list)
for _, row in df_filtered.iterrows():
    image_annotations[row['img_id']].append(row)

print(f"unique images with target species: {len(image_annotations)}")

# build paligemma entries
species_entries = defaultdict(list)
errors = 0

for img_id, annotations in tqdm(image_annotations.items(), desc="building dataset"):
    img_path = available_images[img_id]
    
    try:
        with Image.open(img_path) as img:
            img_w, img_h = img.size
    except:
        errors += 1
        continue
    
    # build suffix with all detections in this image
    suffix_parts = []
    species_in_image = set()
    
    for ann in annotations:
        species_name = TARGET_SPECIES[ann['label_l1']]['name']
        locs = to_paligemma_locs(ann['x_min'], ann['x_max'], ann['y_min'], ann['y_max'], img_w, img_h)
        suffix_parts.append(f"{locs} {species_name}")
        species_in_image.add(ann['label_l1'])
    
    entry = {
        "image": img_path,
        "prefix": "detect fish",
        "suffix": " ; ".join(suffix_parts),
        "species": list(species_in_image)
    }
    
    for sp in species_in_image:
        species_entries[sp].append(entry)

print(f"\nerrors: {errors}")
print(f"\nper-species counts (before balancing):")
for sp, entries in sorted(species_entries.items(), key=lambda x: -len(x[1])):
    print(f"  {sp}: {len(entries)}")

### Classes & Train/Val Split


In [ ]:
# balance: cap each species
print(f"capping at {MAX_SAMPLES_PER_SPECIES} per species...")

balanced = {}  # dedup by image path
for sp, entries in species_entries.items():
    random.shuffle(entries)
    for entry in entries[:MAX_SAMPLES_PER_SPECIES]:
        balanced[entry['image']] = entry

all_entries = list(balanced.values())
random.shuffle(all_entries)
print(f"total unique images after balancing: {len(all_entries)}")

# count whats in the balanced set
from collections import Counter
sp_counts = Counter()
for e in all_entries:
    for sp in e['species']:
        sp_counts[sp] += 1
print("\nbalanced distribution:")
for sp, c in sp_counts.most_common():
    print(f"  {sp}: {c}")

# split
split_idx = int(len(all_entries) * (1 - VAL_SPLIT))
train_entries = all_entries[:split_idx]
val_entries = all_entries[split_idx:]
print(f"\ntrain: {len(train_entries)}")
print(f"val:   {len(val_entries)}")

# save jsonl
train_path = os.path.join(PROCESSED_DIR, "train.jsonl")
val_path = os.path.join(PROCESSED_DIR, "val.jsonl")

for entries, path in [(train_entries, train_path), (val_entries, val_path)]:
    with open(path, 'w') as f:
        for e in entries:
            f.write(json.dumps({"image": e["image"], "prefix": e["prefix"], "suffix": e["suffix"]}) + '\n')

print(f"\nsaved {train_path}")
print(f"saved {val_path}")

# sanity check
print("\n--- first 3 entries ---")
for e in train_entries[:3]:
    print(f"img: {os.path.basename(e['image'])}")
    print(f"suffix: {e['suffix'][:100]}...")
    print()

### Load PaliGemma 2 + QLoRA


In [ ]:
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

print("loading paligemma 2 3b with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = PaliGemmaProcessor.from_pretrained(MODEL_ID)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# freeze vision encoder + projector -- only train language head
for param in model.vision_tower.parameters():
    param.requires_grad = False
for param in model.multi_modal_projector.parameters():
    param.requires_grad = False

# slap lora on
lora_config = LoraConfig(
    r=LORA_RANK,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\nmodel ready lets go")

This collate function will turn our JSONL entries into actual model inputs — loads the image, tokenizes everything, pads the batch.


In [ ]:
from datasets import load_dataset

print("loading datasets...")
train_dataset = load_dataset("json", data_files=train_path, split="train")
val_dataset = load_dataset("json", data_files=val_path, split="train")
print(f"train: {len(train_dataset)} | val: {len(val_dataset)}")

DTYPE = model.dtype

def collate_fn(examples):
    texts = []
    labels = []
    images = []
    
    for ex in examples:
        texts.append(f"<image>{ex['prefix']}")
        labels.append(ex['suffix'])
        try:
            images.append(Image.open(ex['image']).convert('RGB'))
        except:
            # fallback blank image if something breaks
            images.append(Image.new('RGB', (224, 224), color='black'))
    
    tokens = processor(
        text=texts, images=images, suffix=labels,
        return_tensors="pt", padding="longest",
    )
    tokens = tokens.to(DTYPE)
    return tokens

# quick test
print("\ntesting collate with 2 examples...")
test = collate_fn([train_dataset[0], train_dataset[1]])
print(f"batch keys: {list(test.keys())}")
print(f"input_ids shape: {test['input_ids'].shape}")
print("collate works")

### Trainn


In [ ]:
from transformers import Trainer, TrainingArguments

print("=" * 50)
print("STARTING TRAINING")
print(f"epochs: {NUM_EPOCHS}")
print(f"batch: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accumulation = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective")
print(f"lr: {LEARNING_RATE}")
print(f"train size: {len(train_dataset)}")
print("=" * 50)

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    bf16=True,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

print("\n" + "=" * 50)
print("TRAINING DONE")
print("=" * 50)

## 11. Save the LoRA Adapter

This saves just the adapter weights (~45MB). You'll download this to your Mac for inference.


In [ ]:
import shutil

print(f"saving adapter to {ADAPTER_DIR}...")
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

# check size
total = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob("*") if f.is_file())
print(f"adapter size: {total / 1e6:.1f} MB")

# zip it
shutil.make_archive(os.path.join(OUTPUT_DIR, "catchlog-lora-adapter"), 'zip', ADAPTER_DIR)
print(f"zipped to: {OUTPUT_DIR}/catchlog-lora-adapter.zip")
print("\ndownload this zip to your mac!")

## Quick Validation


In [ ]:
import re

def run_inference(model, processor, image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(text="<image>detect fish", images=image, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    return processor.decode(output[0], skip_special_tokens=False)

def parse_detections(text):
    pattern = r'<loc(\d{4})><loc(\d{4})><loc(\d{4})><loc(\d{4})>\s*([^<;]+)'
    return [{'bbox': [int(x1)/1023, int(y1)/1023, int(x2)/1023, int(y2)/1023], 'species': label.strip()}
            for y1, x1, y2, x2, label in re.findall(pattern, text)]

print("testing on validation images...")
print("=" * 50)

correct = 0
total = min(10, len(val_dataset))

for i in range(total):
    entry = val_dataset[i]
    result = run_inference(model, processor, entry['image'])
    detections = parse_detections(result)
    
    # whats expected
    expected = set()
    for sp in TARGET_SPECIES.values():
        if sp['name'] in entry['suffix']:
            expected.add(sp['name'])
    
    # whats predicted  
    predicted = set(d['species'] for d in detections)
    
    match = bool(expected & predicted)
    if match: correct += 1
    
    print(f"\nimage {i+1}: {os.path.basename(entry['image'])}")
    print(f"  expected:  {expected}")
    print(f"  predicted: {predicted}")
    print(f"  {'✓' if match else '✗'}")

print(f"\n{'=' * 50}")
print(f"accuracy: {correct}/{total} ({100*correct/total:.0f}%)")
print(f"{'=' * 50}")